In [1]:
import os

In [2]:
import torch
import torch.nn as nn
import numpy as np
import math

# Force float32
torch.set_default_dtype(torch.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# 1. Model Components
# ----------------------------
class Sine(nn.Module):
    def __init__(self, w0):
        super().__init__()
        self.w0 = w0
    def forward(self, x):
        return torch.sin(self.w0 * x)

class PINN(nn.Module):
    def __init__(self, in_dim=2, width=64, depth=4, out_dim=1, w0=1.137, x0=0.7, v0=1.2):
        super().__init__()
        self.x0, self.v0 = x0, v0
        layers = []
        layers.append(nn.Linear(in_dim, width))
        layers.append(Sine(w0))
        for _ in range(depth - 1):
            layers.append(nn.Linear(width, width))
            layers.append(Sine(w0))
        layers.append(nn.Linear(width, out_dim))
        self.net = nn.Sequential(*layers)
        self.init_weights(w0)

    def init_weights(self, w0):
        is_first = True
        with torch.no_grad():
            for m in self.net:
                if isinstance(m, nn.Linear):
                    in_f = m.weight.size(1)
                    if is_first:
                        m.weight.uniform_(-1 / in_f, 1 / in_f)
                        is_first = False
                    else:
                        bound = math.sqrt(6 / in_f) / w0
                        m.weight.uniform_(-bound, bound)
                    nn.init.zeros_(m.bias)
    def forward(self, z):
        # z: [Batch, 2] -> [t, xi]
        t = z[:, 0:1]
        xi = z[:, 1:2]
        
        # Dimensionless time constant (tau)
        # This ensures the gate and envelope scale together
        tau = xi * t
        envelope = torch.exp(-tau)
        
        # Linear Physics Basis (The "Guess")
        phi = envelope * (self.x0 + (self.v0 + xi * self.x0) * t)
        
        # Adaptive Neural Gate (The "Handover")
        # Using tau here makes the gate 'physics-aware'
        gate = (1.0 - torch.exp(-tau))**2
        
        # Neural Residual
        N = self.net(z)
        
        return phi + gate * N
# ----------------------------
# 2. Physics & Exact Solution
# ----------------------------
def get_exact_solution_torch(t, xi, x0=0.7, v0=1.2):
    # wd = sqrt(1 - xi^2)
    wd = torch.sqrt(torch.clamp(1.0 - xi**2, min=1e-9))
    A = x0
    B = (v0 + xi * x0) / wd
    # Exact: e^(-xi*t) * (A*cos(wd*t) + B*sin(wd*t))
    return torch.exp(-xi * t) * (A * torch.cos(wd * t) + B * torch.sin(wd * t))

def ode_loss(model, z):
    z.requires_grad_(True)
    x = model(z)
    
    # Get dx/dt
    grads = torch.autograd.grad(x, z, torch.ones_like(x), create_graph=True)[0]
    dx_dt = grads[:, 0:1]
    
    # Get d^2x/dt^2
    grads2 = torch.autograd.grad(dx_dt, z, torch.ones_like(dx_dt), create_graph=True)[0]
    ddx_dt2 = grads2[:, 0:1]
    
    xi = z[:, 1:2]
    # Equation: x'' + 2*xi*x' + x = 0
    residual = ddx_dt2 + 2 * xi * dx_dt + x
    return torch.mean(residual**2)

def sample_interior_points(N, T, device):
    t = torch.rand(N, 1, device=device) * T
    xi = 0.1 + (0.4 - 0.1) * torch.rand(N, 1, device=device)
    return torch.cat([t, xi], dim=1)


In [3]:

# # ----------------------------
# # 3. Main Experiment
# # ----------------------------
# time_bounds = [20.0]
# seeds = [0, 1, 2, 3, 4]
# w0 = 1.137

# for T in time_bounds:
#     print(f"\n--- Testing T = {T} ---")
#     num_samples = 3000
#     loss_vals, l2_vals = [], []

#     for seed in seeds:
#         torch.manual_seed(seed)
#         np.random.seed(seed)
        
#         z_i = sample_interior_points(num_samples, T, device)
#         model = PINN(in_dim=2, w0=w0).to(device)

#         # --- Adam Optimization ---
#         optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-4)
#         for _ in range(3000):
#             optimizer_adam.zero_grad()
#             loss = ode_loss(model, z_i)
#             loss.backward()
#             optimizer_adam.step()

#         # --- L-BFGS Optimization ---
#         optimizer_lbfgs = torch.optim.LBFGS(
#             model.parameters(), lr=1.0, max_iter=200, 
#             line_search_fn="strong_wolfe"
#         )
#         def closure():
#             optimizer_lbfgs.zero_grad()
#             l = ode_loss(model, z_i)
#             l.backward()
#             return l
#         optimizer_lbfgs.step(closure)

#         # --- Post-Optimization Logging ---
#         # Calculate final loss BEFORE turning off gradients
#         final_loss = ode_loss(model, z_i).item()
        
#         # --- Global L2 Evaluation ---
#         model.eval()
#         with torch.no_grad():
#             # 1. Create 1D axes on the device
#             t_axis = torch.linspace(0, T, 100, device=device)
#             xi_axis = torch.linspace(0.1, 0.4, 30, device=device)
            
#             # 2. Use cartesian_prod to create the [3000, 2] grid in one shot
#             # This replaces meshgrid, flattening, and stacking.
#             z_test = torch.cartesian_prod(t_axis, xi_axis) # Result is [3000, 2]
            
#             # 3. Predict and Exact in one go
#             x_pred = model(z_test) 
            
#             # Unpack columns for the exact formula
#             t_test_flat = z_test[:, 0:1]
#             xi_test_flat = z_test[:, 1:2]
#             x_true = get_exact_solution_torch(t_test_flat, xi_test_flat)
        
#             # 4. Global L2 Norm (all on GPU)
#             rel_l2 = torch.norm(x_true - x_pred) / torch.norm(x_true)
#             rel_l2_val = rel_l2.item()
        
#         loss_vals.append(final_loss)
#         l2_vals.append(rel_l2_val)
#         model.train() # Switch back for next seed

#     # Statistics Calculation
#     print(f"RESULT | T={T:4.1f} | w0={w0:.3f}")
#     print(f"Loss:  {np.mean(loss_vals):.2e} ± {np.std(loss_vals):.1e}")
#     print(f"RelL2: {np.mean(l2_vals):.2e} ± {np.std(l2_vals):.1e}")

In [4]:
import numpy as np
import torch

# 1. Define range for w0: 15 values log-spaced from 0.01 to 4
w0_values = np.linspace(3.75, 5, num = 10)
time_bounds = [20.0]
seeds = [0]

for T in time_bounds:
    print(f"\n{'='*30} Testing T = {T} {'='*30}")
    
    for w0 in w0_values:
        print(f"\n>> Configuration: w0 = {w0:.4f}")
        num_samples = 7500
        adam_losses, final_losses, l2_vals = [], [], []

        for seed in seeds:
            torch.manual_seed(seed)
            np.random.seed(seed)
            
            # Ensure z_i requires grad for the ODE derivatives
            z_i = sample_interior_points(num_samples, T, device).requires_grad_(True)
            model = PINN(in_dim=2, w0=w0).to(device)

            # --- Phase 1: Adam Optimization ---
            optimizer_adam = torch.optim.Adam(model.parameters(), lr=1e-3)
            for _ in range(4000):
                optimizer_adam.zero_grad()
                loss = ode_loss(model, z_i)
                loss.backward()
                optimizer_adam.step()
            
            # CORRECTED: Calculate Adam loss WITHOUT no_grad() 
            # because ode_loss needs the autograd graph for derivatives.
            current_adam_err = ode_loss(model, z_i).item()
            adam_losses.append(current_adam_err)

            # --- Phase 2: L-BFGS Optimization ---
            optimizer_lbfgs = torch.optim.LBFGS(
                model.parameters(), lr=1.0, max_iter=800, history_size=200, tolerance_grad=1e-100, tolerance_change=1e-100, 
                line_search_fn="strong_wolfe"
            )
            def closure():
                optimizer_lbfgs.zero_grad()
                l = ode_loss(model, z_i)
                l.backward()
                return l
            optimizer_lbfgs.step(closure)

            # CORRECTED: Calculate Final loss WITHOUT no_grad()
            current_final_err = ode_loss(model, z_i).item()
            final_losses.append(current_final_err)
            
            # --- Global L2 Evaluation ---
            # This is safe for no_grad() because it's a value comparison
            model.eval()
            with torch.no_grad():
                t_axis = torch.linspace(0, T, 100, device=device)
                xi_axis = torch.linspace(0.1, 0.4, 30, device=device)
                z_test = torch.cartesian_prod(t_axis, xi_axis)
                
                x_pred = model(z_test) 
                x_true = get_exact_solution_torch(z_test[:, 0:1], z_test[:, 1:2])
            
                rel_l2 = torch.norm(x_true - x_pred) / torch.norm(x_true)
                l2_vals.append(rel_l2.item())
            
            model.train()

        # Final logs for this specific w0
        print(f"Results for w0={w0:.4f}:")
        print(f"  Avg Adam Error : {np.mean(adam_losses):.4e} ± {np.std(adam_losses):.1e}")
        print(f"  Avg Final Error: {np.mean(final_losses):.4e} ± {np.std(final_losses):.1e}")
        print(f"  Avg RelL2 Error: {np.mean(l2_vals):.4e} ± {np.std(l2_vals):.1e}")


============================== Testing T = 20.0 ==============================

>> Configuration: w0 = 3.7500
Results for w0=3.7500:
  Avg Adam Error : 6.2009e-03 ± 0.0e+00
  Avg Final Error: 1.4801e-04 ± 0.0e+00
  Avg RelL2 Error: 7.0939e-02 ± 0.0e+00

>> Configuration: w0 = 3.8889
Results for w0=3.8889:
  Avg Adam Error : 1.1278e-02 ± 0.0e+00
  Avg Final Error: 3.1941e-04 ± 0.0e+00
  Avg RelL2 Error: 1.0166e-01 ± 0.0e+00

>> Configuration: w0 = 4.0278
Results for w0=4.0278:
  Avg Adam Error : 7.8284e-03 ± 0.0e+00
  Avg Final Error: 2.7864e-04 ± 0.0e+00
  Avg RelL2 Error: 1.0198e-01 ± 0.0e+00

>> Configuration: w0 = 4.1667
Results for w0=4.1667:
  Avg Adam Error : 1.0321e-02 ± 0.0e+00
  Avg Final Error: 3.3625e-04 ± 0.0e+00
  Avg RelL2 Error: 1.3341e-01 ± 0.0e+00

>> Configuration: w0 = 4.3056
Results for w0=4.3056:
  Avg Adam Error : 6.4349e-03 ± 0.0e+00
  Avg Final Error: 3.5991e-04 ± 0.0e+00
  Avg RelL2 Error: 1.5269e-01 ± 0.0e+00

>> Configuration: w0 = 4.4444
Results for w0=4.44